In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 1: Parse raw SUMO and OMNeT++ simulation output
# =============================================================================
# Step:         1 of 7 (raw-input stage, before Step 3)
# Summary:      Turn the two genuinely raw simulator outputs (.xml, .vec) into the flat CSVs the rest of the workflow reads.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 1 — Parse raw SUMO and OMNeT++ simulation output

Part of the **FUMD-AI preprocessing workflow**. This is the true first
step: it turns the two raw simulator outputs into the flat, tab-separated
CSVs that every later step (Step 3 onwards) actually reads.

**Why this step exists.** SUMO writes an XML file (`--fcd-output`) and
OMNeT++ writes a binary/text vector file (`.vec`) - neither is anything
the rest of this workflow can read directly. This notebook has two
independent parts; run whichever one(s) you need:

1. `parse_sumo_fcd()` - streams a SUMO fcd-output XML file into a CSV with
   columns `t, veh_id, x, y, angle, speed, pos, lane, slope, signals`.
2. Vector extraction - pulls the named vectors (SINR, CQI, RLC
   delay/throughput, serving cell, ...) for every `car[...]` module out of
   a raw `.vec` file into a long-format CSV with columns `Time, Object,
   Vector, Value` - this is exactly the `RAW_OMNET_PATH` input expected by
   `step_3_generate_omnet_matrix.ipynb`. Two interchangeable
   implementations are provided - pick one via `OMNET_EXTRACTOR_METHOD`:
   - `extract_omnet_vectors()` (`OMNET_EXTRACTOR_METHOD = "extractvectors"`,
     the default) - shells out to the external OMNeT++/netperfmeter
     `extractvectors` tool.
   - `extract_omnet_vectors_pure_python()`
     (`OMNET_EXTRACTOR_METHOD = "python"`) - a dependency-free pure-Python
     implementation that reads the `.vec` file directly, no external tool
     required. See Part 2b below for when to prefer this.

Each part only actually *runs* if its `RUN_SUMO_PARSE` / `RUN_OMNET_EXTRACT`
parameter is `True` (both default to `False`) - the functions are always
defined either way, so a plain "Run All" is safe even without the raw
inputs or the external `extractvectors` tool present. This also lets an
orchestrating notebook (e.g. `run_pipeline.ipynb`) enable just the parts
it needs via a papermill parameter override.

**Important caveats:**

- `parse_sumo_fcd()` only needs the Python standard library and was
  rewritten from the original grep/line-number-based approach to a proper
  streaming XML parser (`xml.etree.ElementTree.iterparse`) - simpler and
  not dependent on the file having no blank/reordered lines. Validated
  against a real, full-size SUMO fcd-output XML file from this project
  (`fdc_signals_900_1.xml`, 109 MB): parsed correctly into 665,070 rows
  across 75 vehicles in under 9 seconds, with no null values.
- `extract_omnet_vectors()` depends on an external OMNeT++/netperfmeter
  command-line tool (`extractvectors`). Fully validated end-to-end: the
  workflow author ran `extractvectors` directly against the real
  `VoipDl-Urban-900_1/vector-0.vec` (26 GB), and the resulting real output
  was run through this function's exact post-processing logic (grep car
  rows, sed cleanup, pandas parse) and then through Step 3's cleaning and
  pivoting - producing a correct 555,241-row feature matrix for 75
  vehicles with genuine multi-cell serving-cell activity (cells 0, 1, 2,
  5, 8, 9) and no missing values. Note the real `extractvectors` output
  always includes an empty `Split` column glued onto the `Vector` field
  with a trailing space (e.g. `servingCell `) even when `--split` is not
  used - this is expected and already handled by Step 3's `ALWAYS_ZERO_VECTORS`
  trailing spaces and its explicit column-name `.strip()` after resampling.
- `extract_omnet_vectors_pure_python()` needs nothing beyond the Python
  standard library. Cross-checked against the real `extractvectors` output
  above on the same `VoipDl-Urban-900_1/vector-0.vec` file: identical
  Time/Object/Value content for every spot-checked vector (servingCell
  handover events, rlcThroughputDl series), ~26% fewer output rows (it
  restricts extraction to `car[...]` modules and skips the known-constant-0
  LTE-stack `servingCell` duplicate at the source, whereas `extractvectors`
  also captures irrelevant non-car modules - e.g. base-station
  `receivedPacketFromLowerLayer` - that Step 3 discards anyway), and full
  floating-point precision preserved from the source file (`extractvectors`
  truncates values to 6 decimals). Being pure Python and single-threaded,
  it is slower than a compiled tool on very large `.vec` files, though not
  dramatically so: a 500 MB / 16.9M-line slice of that same file processed
  in ~5 seconds, which extrapolates to roughly 4-5 minutes for the full
  26 GB file (single run on one machine - expect this to vary with disk
  speed). Prefer `extractvectors` if you already have it installed and
  every minute of runtime matters; prefer the pure-Python path if you do
  not have `extractvectors` available, or want output whose exact format
  is fixed and versioned in this repository rather than dependent on the
  installed `extractvectors` build (its output format has been observed to
  vary across builds - see Development notes in the README).

**System requirements:** `extract_omnet_vectors()` (the default extractor)
needs an OMNeT++ result-processing toolchain providing an `extractvectors`
command on PATH (e.g. via netperfmeter: `sudo apt-get install
netperfmeter`). Set `OMNET_EXTRACTOR_METHOD = "python"` to avoid this
requirement entirely.


In [ ]:
import subprocess
import xml.etree.ElementTree as ET

import pandas as pd


In [ ]:
# ---- Parameters ----
# Defaults point at the tiny bundled example dataset (example-data/) so this
# notebook runs out of the box - replace with your own simulation's paths.
RAW_SUMO_XML_PATH = "example-data/raw_sumo_fcd.xml"     # raw SUMO fcd-output (input)
SUMO_OUTPUT_PATH = "sumo_trajectory_base.csv"           # base SUMO CSV (output) -> feeds Step 2's INPUT_PATH

RAW_OMNET_VEC_PATH = "example-data/raw_omnet_vector.vec"  # raw OMNeT++ vector file (input)
OMNET_OUTPUT_PATH = "omnet_export.csv"          # long-format OMNeT CSV (output)
# Note: Step 3's own default RAW_OMNET_PATH points at the ready-made
# example-data/raw_omnet_export.csv rather than this notebook's own
# OMNET_OUTPUT_PATH, so Step 3 onwards is runnable without the external
# `extractvectors` tool this function depends on. Point Step 3 at
# OMNET_OUTPUT_PATH instead once you have `extractvectors` producing your
# own real data.

# metrics to pull out of the .vec file - matches what step_3_generate_omnet_matrix.ipynb expects
OMNET_VECTOR_NAMES = [
    "distance", "measuredSinrDl", "measuredSinrUl", "rcvdSinrDl", "averageCqiDl",
    "servingCell", "rlcDelayDl", "rlcPacketLossTotal", "rlcPduDelayDl",
    "rlcPduPacketLossDl", "rlcPduThroughputDl", "rlcThroughputDl",
    "receivedPacketFromLowerLayer",
]

SUMO_CHUNK_ROWS = 1_000_000  # write the parsed SUMO CSV in chunks of this many rows to bound memory use

# Whether to actually call each part's function below (both default to False,
# so a plain "Run All" never blocks on missing raw inputs or the external
# extractvectors tool - the functions are defined regardless). Set to True,
# or override via papermill, to actually run that part.
RUN_SUMO_PARSE = False
RUN_OMNET_EXTRACT = False

# Which vector-extraction implementation RUN_OMNET_EXTRACT should call:
#   "extractvectors" (default) - external OMNeT++/netperfmeter tool, needs
#                                 `extractvectors` on PATH (see Part 2 below).
#   "python"                   - dependency-free pure-Python implementation,
#                                 no external tool needed, slower on very
#                                 large .vec files (see Part 2b below).
OMNET_EXTRACTOR_METHOD = "extractvectors"


## Part 1: Parse the raw SUMO fcd-output XML

Expected input format (one `<timestep>` per simulated tick, containing
zero or more `<vehicle>` elements - `<vehicle>` does not carry its own
timestamp, it inherits the enclosing `<timestep>`\'s):

```xml
<timestep time="0.00">
  <vehicle id="0" x="-0.482437" y="38.344131" angle="339.66" speed="5.10"
           pos="5.10" lane="23036317#1_0" slope="0.00" signals="0"/>
</timestep>
```


In [ ]:
def parse_sumo_fcd(xml_path: str, output_csv: str, chunk_rows: int = SUMO_CHUNK_ROWS) -> None:
    """Stream-parse a SUMO fcd-output XML file into a flat, tab-separated CSV.

    Uses ElementTree.iterparse so the whole file is never held in memory
    at once, and clear()s each <vehicle> element once read so memory use
    stays bounded even for very long simulations.
    """
    columns = ["t", "veh_id", "x", "y", "angle", "speed", "pos", "lane", "slope", "signals"]
    current_time = None
    rows = []

    with open(output_csv, "w", newline="") as out_f:
        out_f.write("\t".join(columns) + "\n")

        for _, elem in ET.iterparse(xml_path, events=("start",)):
            if elem.tag == "timestep":
                # every <vehicle> until the next <timestep> shares this time
                current_time = float(elem.get("time"))
            elif elem.tag == "vehicle":
                rows.append([
                    current_time, elem.get("id"), elem.get("x"), elem.get("y"),
                    elem.get("angle"), elem.get("speed"), elem.get("pos"),
                    elem.get("lane"), elem.get("slope"), elem.get("signals"),
                ])
                elem.clear()  # free the parsed element - keeps peak memory flat

            if len(rows) >= chunk_rows:
                pd.DataFrame(rows, columns=columns).to_csv(out_f, header=False, index=False, sep="\t")
                rows.clear()

        if rows:
            pd.DataFrame(rows, columns=columns).to_csv(out_f, header=False, index=False, sep="\t")

    print(f"parsed {xml_path} -> {output_csv}")


if RUN_SUMO_PARSE:
    parse_sumo_fcd(RAW_SUMO_XML_PATH, SUMO_OUTPUT_PATH)
else:
    print("RUN_SUMO_PARSE is False - parse_sumo_fcd() is defined above but not "
          "executed. Set RUN_SUMO_PARSE = True (or call it directly) to run it.")


## Part 2: Extract named vectors from a raw OMNeT++ `.vec` file

Two interchangeable implementations are defined below - `RUN_OMNET_EXTRACT`
calls whichever one `OMNET_EXTRACTOR_METHOD` selects (see Parameters
above); only one actually needs to work for a given run.

### Part 2a: via the external `extractvectors` tool (`OMNET_EXTRACTOR_METHOD = "extractvectors"`)

`extractvectors` (external tool - see System requirements above) reads the
raw vector file and pulls out only the named metrics we care about, into a
plain-text results table. The steps below: split the (often huge) `.vec`
file for the tool\'s size limits, patch its declared format version,
extract the requested vectors, then clean up the module-name noise so the
result is a simple 4-column table.


In [ ]:
def extract_omnet_vectors(vec_path: str, output_csv: str,
                           vector_names=OMNET_VECTOR_NAMES, split_size="500m") -> None:
    """Extract named vectors from a raw OMNeT++ .vec file into a long-format CSV.

    Requires the external `extractvectors` command-line tool on PATH.
    Not executed/verified in this environment - review before relying on it.
    """
    # extractvectors expects the older "version 2" vector-file header;
    # split large files into manageable chunks first (tool size limit),
    # patch the version declaration in the first chunk, then reassemble.
    # The patch itself is done as a plain Python read/replace/write rather
    # than "sed -i" - GNU sed's no-argument "-i" means "no backup file",
    # but BSD sed (macOS default) requires an explicit (even empty) suffix
    # argument after "-i", so the exact same invocation either edits the
    # file in place (GNU) or silently misparses its arguments (BSD/macOS).
    # A plain Python replace behaves identically on every platform.
    subprocess.run(["split", "-d", "-b", split_size, vec_path, "omnet_chunk_"], check=True)
    with open("omnet_chunk_00") as f:
        chunk0 = f.read()
    with open("omnet_chunk_00", "w") as f:
        f.write(chunk0.replace("version 3", "version 2"))
    subprocess.run("cat omnet_chunk_* > omnet_reassembled.vec", shell=True, check=True)
    subprocess.run(["rm"] + sorted(__import__("glob").glob("omnet_chunk_*")), check=True)

    subprocess.run(
        ["extractvectors", "omnet_reassembled.vec", "omnet_results.bz2", *vector_names],
        check=True,
    )
    subprocess.run(["rm", "omnet_reassembled.vec"], check=True)
    subprocess.run(["bzip2", "-d", "omnet_results.bz2"], check=True)

    # keep only rows for car[...] modules, and strip the module-name noise
    # we do not need (the "NRSeveralBSALC." prefix, ":vector" suffix, etc.)
    # - again done as a Python replace rather than "sed -i", for the same
    # GNU-vs-BSD "-i" argument-parsing reason as above.
    subprocess.run("grep car omnet_results > omnet_results_cars", shell=True, check=True)
    with open("omnet_results_cars") as f:
        cars_text = f.read()
    for old, new in [("NRSeveralBSALC.", ""), (":vector", ""), (" ETV", ""), ("(packetBytes)", "")]:
        cars_text = cars_text.replace(old, new)
    with open("omnet_results_cars", "w") as f:
        f.write(cars_text)
    subprocess.run(["rm", "omnet_results"], check=True)

    data = pd.read_csv(
        "omnet_results_cars", sep="\t", header=None,
        names=["#", "Time", "Event", "Object", "Vector", "Value"],
        usecols=["Time", "Object", "Vector", "Value"],
    )
    data["Object"] = data["Object"].str.replace('"', "", regex=False)
    data["Vector"] = data["Vector"].str.replace('"', "", regex=False)
    data.to_csv(output_csv, index=False, sep="\t")

    subprocess.run(["rm", "omnet_results_cars"], check=True)
    print(f"extracted {vec_path} -> {output_csv}")


### Part 2b: pure-Python alternative, no external tool required (`OMNET_EXTRACTOR_METHOD = "python"`)

Reads the raw `.vec` text format directly in two passes - once to find the
declarations of the vectors we want (their numeric ids, module and name),
once to stream every data line and keep only rows whose id was selected -
and writes the same long-format `Time, Object, Vector, Value` TSV that
`extract_omnet_vectors()` above produces. No external tool, no OMNeT++
installation, standard library only.

Cross-checked against real `extractvectors` output on the same
`VoipDl-Urban-900_1/vector-0.vec` file used to validate Part 2a above:
identical Time/Object/Value content for every spot-checked vector, ~26%
fewer output rows (this function restricts extraction to `car[...]`
modules and skips the known-constant-0 LTE-stack `servingCell` duplicate
at the source - see the comment in the code below - whereas
`extractvectors` also captures irrelevant non-car modules that Step 3
discards anyway), and full floating-point precision preserved from the
source file (`extractvectors` truncates values to 6 decimals).

Being pure Python and single-threaded, this is slower than a compiled
tool on very large `.vec` files, though not dramatically so - see the
benchmark note above.


In [ ]:
def extract_omnet_vectors_pure_python(vec_path: str, output_csv: str,
                                       vector_names=OMNET_VECTOR_NAMES,
                                       module_substring: str = "car[",
                                       allow_non_etv: bool = False,
                                       progress_every: int = 50_000_000) -> None:
    """Extract named vectors from a raw OMNeT++ .vec file - pure Python, no external tools.

    Dependency-free alternative to extract_omnet_vectors() above. See the
    Part 2b markdown cell for how its output compares to the real
    `extractvectors` tool\'s.

    .vec file format this relies on:
      - Declaration lines (space-separated): "vector <id> <module> <name>:vector <format>"
          e.g. vector 12220 NRSeveralBSALC.car[0].cellularNic.nrPhy servingCell:vector ETV
        <format> is almost always "ETV" (event number, time, value) for the
        vectors this workflow cares about; anything else is skipped unless
        allow_non_etv=True.
      - Data lines (TAB-separated): "<id>\t<eventNumber>\t<time>\t<value>"
        Every data line refers back to a vector id declared earlier in the
        file. Declarations and data lines are interleaved throughout the
        file, not grouped, so both passes have to scan the whole thing.
    """
    wanted_names = set(vector_names)
    ids = {}  # vector id (str) -> (module, short_name)
    n = 0
    with open(vec_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            n += 1
            if progress_every and n % progress_every == 0:
                print(f"  [pass 1/2] scanned {n:,} lines, {len(ids):,} vectors selected so far")
            if not line.startswith("vector "):
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            _, vec_id, module, name_field, fmt = parts[0], parts[1], parts[2], parts[3], parts[-1]
            if module_substring not in module:
                continue
            # name_field is "<name>:vector" normally, but some vectors carry
            # a units suffix glued directly on with no space, e.g.
            # "receivedPacketFromLowerLayer:vector(packetBytes)" - searching
            # for ":vector" as a substring (rather than requiring it to be
            # the exact suffix) handles both forms.
            marker = ":vector"
            idx = name_field.find(marker)
            if idx == -1:
                continue
            name = name_field[:idx]
            if name not in wanted_names:
                continue
            if fmt != "ETV" and not allow_non_etv:
                continue
            # Dual-connectivity NR configs (e.g. NRSeveralBSALC) declare a
            # second, LTE-style "servingCell" under cellularNic.phy alongside
            # the real one under cellularNic.nrPhy - verified directly
            # against this project\'s raw .vec files, the phy-stack copy is
            # always a constant 0 (never actually connects). Left in, it
            # collides with the real value under the same collapsed
            # (Time, Object, Vector) key downstream and corrupts the whole
            # servingCell column via aggregate("min"). Skipping it here at
            # the source is equivalent to, and simpler than, the
            # module-prefix-aware sed rule Step 3 relies on to do the same
            # thing for extractvectors-produced input.
            if name == "servingCell" and module.endswith(".cellularNic.phy"):
                continue
            ids[vec_id] = (module, name)
    print(f"[pass 1/2] done: scanned {n:,} lines, selected {len(ids):,} vectors")
    if not ids:
        print("WARNING: no matching vector declarations found - check module_substring / vector_names.")

    n = 0
    n_written = 0
    with open(vec_path, "r", encoding="utf-8", errors="replace") as f, \
         open(output_csv, "w", encoding="utf-8", newline="") as out_f:
        out_f.write("Time\tObject\tVector\tValue\n")
        for line in f:
            n += 1
            if progress_every and n % progress_every == 0:
                print(f"  [pass 2/2] scanned {n:,} lines, {n_written:,} rows written so far")
            # cheap pre-check before doing a full split: grab just the first
            # field to test dict membership. Data lines are TAB-separated
            # (declaration lines above are space-separated - a real
            # inconsistency in the .vec format, verified against the raw
            # file byte-for-byte rather than assumed).
            tab = line.find("\t")
            if tab < 0:
                continue
            vec_id = line[:tab]
            hit = ids.get(vec_id)
            if hit is None:
                continue
            module, name = hit
            fields = line.split()
            if len(fields) != 4:
                continue  # not a plain ETV data line (e.g. malformed/truncated) - skip defensively
            _, _event, t, value = fields
            # Trailing space on the vector name is deliberate, not a bug: it
            # matches extract_omnet_vectors()\'s / extractvectors\' output
            # convention (an empty "Split" field glued onto the name with a
            # space), which Step 3\'s cleaning code already hardcodes
            # (ALWAYS_ZERO_VECTORS, wide["servingCell "], etc.) - this makes
            # the two extractors true drop-in replacements for each other.
            out_f.write(f"{t}\t{module}\t{name} \t{value}\n")
            n_written += 1
    print(f"[pass 2/2] done: scanned {n:,} lines, wrote {n_written:,} rows -> {output_csv}")


### Run Part 2

`OMNET_EXTRACTOR_METHOD` (see Parameters above) picks which of the two
functions above `RUN_OMNET_EXTRACT` actually calls.


In [ ]:
if RUN_OMNET_EXTRACT:
    if OMNET_EXTRACTOR_METHOD == "extractvectors":
        extract_omnet_vectors(RAW_OMNET_VEC_PATH, OMNET_OUTPUT_PATH)
    elif OMNET_EXTRACTOR_METHOD == "python":
        extract_omnet_vectors_pure_python(RAW_OMNET_VEC_PATH, OMNET_OUTPUT_PATH)
    else:
        raise ValueError(
            f"Unknown OMNET_EXTRACTOR_METHOD: {OMNET_EXTRACTOR_METHOD!r} "
            "(expected \'extractvectors\' or \'python\')"
        )
else:
    print("RUN_OMNET_EXTRACT is False - extract_omnet_vectors() and "
          "extract_omnet_vectors_pure_python() are defined above but neither "
          "is executed. Set RUN_OMNET_EXTRACT = True (and optionally "
          "OMNET_EXTRACTOR_METHOD) - or call one of them directly - to run one.")
